In [1]:
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats.mstats import winsorize
plt.style.use('seaborn-v0_8-whitegrid')
def pivot(variable):
    return raw_df.pivot(index='year', columns='country', values=variable)
def add_const(df):
    return sm.add_constant(df)
def melt(df):
    return df.melt(ignore_index=False).reset_index().set_index(['country','year'])
def winsor(df):
    return df.apply(lambda x: winsorize(x, limits=(.05,.05)))

In [2]:
def significance_stars(pval):
    if pval < 0.01:
        return '***'
    elif pval < 0.05:
        return '**'
    elif pval < 0.1:
        return '*'
    else:
        return ''

def display(fit):
    df = pd.DataFrame({'coefficient': fit.params, 'pvalue': fit.pvalues})
    df['significance'] = df['pvalue'].apply(significance_stars)
    df['results'] = df.apply(
        lambda x: f"{x['coefficient']:.3f}{x['significance']}", axis=1)
    return df[['results']]

def interaction_display(fit):
    result = display(fit)[1:3+1]
    # if (result.iloc[1][0][-2:] == '**'):
    #     result.loc['interaction'] = '('+result.loc['interaction']+')'
    return result

def column_to_df(column):
    df = cpi_df.copy()
    df[:] = np.nan
    for country in df.columns:
        df[country] = column
    return df

## Data Import

In [5]:
raw_df = pd.read_excel('datasets/JSTdatasetR6.xlsx').drop(columns=['country','ifs']).rename(columns={'iso':'country'})
raw_oil_shocks_df = pd.read_excel('datasets/BH2_supply_shocks.xlsx',header=1,usecols=[1])
credibility_df = pd.read_excel('datasets/CBIData_Romelli_2024.xlsx',
                               sheet_name=1,usecols=['year','iso_a3','cbie_index', 'cbie_policy', 'cbie_finindep']).rename(columns={'iso_a3':'country'})
raw_df = pd.merge(raw_df, credibility_df, how='left',on=['year','country'])


## Data Cleaning

In [17]:
cpi_df = pivot('cpi')
gdp_df = pivot('rgdpbarro')
stir_df = pivot('stir')
crisis_df = pivot('crisisJST')
# ------------------------------------------------------------------------------------------------------------
fx_rate_df = pivot('xrusd')
peg_df = pivot('peg')
crisis_old_df = pivot('crisisJST_old')
# ------------------------------------------------------------------------------------------------------------
raw_oil_shocks_df.index = pd.date_range(start='1975-02-01', periods=len(raw_oil_shocks_df), freq='ME')
raw_oil_shocks_df = raw_oil_shocks_df.reindex(pd.date_range(start='1975-01-01', periods=len(raw_oil_shocks_df)+1,freq='ME'))
oil_shocks = raw_oil_shocks_df.resample('YE').mean()*12
oil_shocks.index = range(1975,1975+len(oil_shocks))
oil_shocks_df  = cpi_df.copy()
oil_shocks_df[:] = np.nan
for country in oil_shocks_df.columns:
    oil_shocks_df[country] = oil_shocks['oil supply shocks']
# -----------------------------------------------------------------------
credit_growth_df = pivot('tloans')/cpi_df
for country in credit_growth_df.columns:
    credit_growth_df[country] = np.log(credit_growth_df[country])
credit_growth_df = credit_growth_df.diff()
#--------------------------------------------------------------------------------
mortgage_to_gdp_df = winsor(pivot('tmort')/pivot('gdp'))
corporate_debt_to_gdp_df = winsor(pivot('bdebt')/pivot('gdp'))
public_debt_df = pivot('debtgdp')

#--------------------------------------------------------------------------------
credibility_df = df = pivot('cbie_index')
credibility_relative_df = credibility_df.sub(credibility_df.median(axis=1),axis=0)
credibility_above_average_df = (credibility_relative_df>0).astype(int)
credibility_of_policy_df = df = pivot('cbie_policy')
credibility_relative_of_policy_df = credibility_of_policy_df.sub(credibility_of_policy_df.median(axis=1),axis=0)
credibility_of_policy_above_average_df = (credibility_relative_of_policy_df>0).astype(int)
credibility_of_fin_indep_df = df = pivot('cbie_finindep')
credibility_relative_of_fin_indep_df = credibility_of_fin_indep_df.sub(credibility_of_fin_indep_df.median(axis=1),axis=0)
credibility_of_fin_indep_above_average_df = (credibility_relative_of_fin_indep_df>0).astype(int)
# -------------------------------------
wages_df = pivot('wage')


# Regression

In [ ]:

panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values

crises_across_horizon_df = (crisis_df==1)|(crisis_df.shift(-1)==1)|(crisis_df.shift(-2)==1)
panel_df['crisis'] = melt(crisis_df)

covariates = []
for lag in range(4+1):
    panel_df['stir(-'+str(lag)+')'] = melt(stir_df.diff().shift(lag))
    panel_df['gdp(-'+str(lag)+')'] = melt((gdp_df.pct_change(fill_method=None)*100).shift(lag))
    covariates.append('stir(-'+str(lag)+')')
    covariates.append('gdp(-'+str(lag)+')')
    
first_stage_covariates = ['oil_shock'] + covariates
panel_df['infl'] = melt(winsor((cpi_df.pct_change()*100).diff()))
panel_df['oil_shock'] = melt(oil_shocks_df.shift(0))
sub_sample = panel_df[(panel_df.year > 1974)]#&(panel_df.year < 2000)]
result = PanelOLS(sub_sample['infl'], add_const(sub_sample[first_stage_covariates]),entity_effects=True, time_effects=False
                  ).fit(cov_type="clustered", cluster_entity=True)
fitted_df = result.fitted_values.reset_index().pivot(index='year',columns='country',values='fitted_values')

parameter = winsor(wages_df.pct_change()*100).shift(2)

panel_df['delta_inflation_hat'] = melt(fitted_df.shift(1))
panel_df['inflation X wages'] = melt(parameter*fitted_df.shift(1))
panel_df['wages'] = melt(parameter)
second_stage_covariates = ['delta_inflation_hat', 'inflation X wages', 'wages'] + covariates

sub_sample = panel_df[(panel_df.year > 1974)]
fit = PanelOLS(sub_sample['crisis'], add_const(sub_sample[second_stage_covariates]), entity_effects=True,
               time_effects=False).fit(cov_type="clustered", cluster_entity=True)
display(fit)

In [18]:

panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values

crises_across_horizon_df = (crisis_df==1)|(crisis_df.shift(-1)==1)|(crisis_df.shift(-2)==1)
panel_df['crisis'] = melt(crisis_df)

covariates = []
for lag in range(4+1):
    panel_df['stir(-'+str(lag)+')'] = melt(stir_df.diff().shift(lag))
    panel_df['gdp(-'+str(lag)+')'] = melt((gdp_df.pct_change(fill_method=None)*100).shift(lag))
    covariates.append('stir(-'+str(lag)+')')
    covariates.append('gdp(-'+str(lag)+')')
    
first_stage_covariates = ['oil_shock'] + covariates
panel_df['infl'] = melt(winsor((cpi_df.pct_change()*100).diff()))
panel_df['oil_shock'] = melt(oil_shocks_df.shift(0))
sub_sample = panel_df[(panel_df.year > 1974)]#&(panel_df.year < 2000)]
result = PanelOLS(sub_sample['infl'], add_const(sub_sample[first_stage_covariates]),entity_effects=True, time_effects=False
                  ).fit(cov_type="clustered", cluster_entity=True)
fitted_df = result.fitted_values.reset_index().pivot(index='year',columns='country',values='fitted_values')

parameter = (credibility_relative_of_fin_indep_df).shift(2)

panel_df['delta_inflation_hat'] = melt(fitted_df.shift(1))
panel_df['inflation X credibility_of_fin_indep'] = melt(parameter*fitted_df.shift(1))
panel_df['credibility_of_fin_indep'] = melt(parameter)
second_stage_covariates = ['delta_inflation_hat', 'inflation X credibility_of_fin_indep', 'credibility_of_fin_indep'] + covariates

sub_sample = panel_df[(panel_df.year > 1974)]
fit = PanelOLS(sub_sample['crisis'], add_const(sub_sample[second_stage_covariates]), entity_effects=True,
               time_effects=False).fit(cov_type="clustered", cluster_entity=True)
display(fit)

c:\Users\james\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\james\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


,results
const,0.047***
delta_inflation_hat,0.029**
inflation X credibility_of_fin_indep,-0.059
credibility_of_fin_indep,0.025
stir(-0),0.005*
gdp(-0),-0.009***
stir(-1),-0.007
gdp(-1),0.004*
stir(-2),0.004
gdp(-2),-0.000


In [19]:

panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values

crises_across_horizon_df = (crisis_df==1)|(crisis_df.shift(-1)==1)|(crisis_df.shift(-2)==1)
panel_df['crisis'] = melt(crisis_df)

covariates = []
for lag in range(4+1):
    panel_df['stir(-'+str(lag)+')'] = melt(stir_df.diff().shift(lag))
    panel_df['gdp(-'+str(lag)+')'] = melt((gdp_df.pct_change(fill_method=None)*100).shift(lag))
    covariates.append('stir(-'+str(lag)+')')
    covariates.append('gdp(-'+str(lag)+')')
    
first_stage_covariates = ['oil_shock'] + covariates
panel_df['infl'] = melt(winsor((cpi_df.pct_change()*100).diff()))
panel_df['oil_shock'] = melt(oil_shocks_df.shift(0))
sub_sample = panel_df[(panel_df.year > 1974)]#&(panel_df.year < 2000)]
result = PanelOLS(sub_sample['infl'], add_const(sub_sample[first_stage_covariates]),entity_effects=True, time_effects=False
                  ).fit(cov_type="clustered", cluster_entity=True)
fitted_df = result.fitted_values.reset_index().pivot(index='year',columns='country',values='fitted_values')

panel_df['delta_inflation_hat'] = melt(fitted_df.shift(1))

panel_df['public_debt'] = melt(public_debt_df.shift(2))
panel_df['mortgages'] = melt(mortgage_to_gdp_df.shift(2))
panel_df['corporate_debt'] = melt(corporate_debt_to_gdp_df.shift(2))

panel_df['delta_inflation_hat X public_debt'] = melt(fitted_df.shift(1) * public_debt_df.shift(2))
panel_df['delta_inflation_hat X mortgages'] = melt(fitted_df.shift(1) * mortgage_to_gdp_df.shift(2))
panel_df['delta_inflation_hat X corporate_debt'] = melt(fitted_df.shift(1) * corporate_debt_to_gdp_df.shift(2))

second_stage_covariates = ['delta_inflation_hat', 'public_debt', 'mortgages',
                           'corporate_debt', 'delta_inflation_hat X public_debt',
                            'delta_inflation_hat X mortgages', 'delta_inflation_hat X corporate_debt'] + covariates

sub_sample = panel_df[(panel_df.year > 1974)]
fit = PanelOLS(sub_sample['crisis'], add_const(sub_sample[second_stage_covariates]), entity_effects=True,
               time_effects=False).fit(cov_type="clustered", cluster_entity=True)
display(fit)

c:\Users\james\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\james\anaconda3\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


,results
const,-0.013
delta_inflation_hat,-0.022
public_debt,-0.021
mortgages,0.062
corporate_debt,0.042
delta_inflation_hat X public_debt,0.020
delta_inflation_hat X mortgages,0.084***
delta_inflation_hat X corporate_debt,0.005
stir(-0),0.005*
gdp(-0),-0.008***
